In [1]:
from openai import OpenAI
from rich.console import Console
from dotenv import load_dotenv
import json

In [2]:
load_dotenv(override=True)

True

In [3]:
todo=[]
complete=[]

In [4]:
def show(message):
    try:
        Console().print(message)
    except:
        print(message)

In [5]:
def get_report_todo()->str:
    list_todo=""
    for index,task in enumerate(todo):
        if complete[index]:
            list_todo+=f"Step {index+1}: [green][strike]{task}[/strike][/green]\n"
        else:
            list_todo+=f"Step {index+1}: {task}\n"
    show(list_todo)
    return list_todo

In [6]:
def create_todo_list(list_task:list[str]):
    todo.extend(list_task)
    complete.extend([False]*len(todo))
    return get_report_todo()

In [7]:
def mark_complete(index,completion_notes):
    if 1<= index <=len(todo):
        complete[index-1]=True
    else:
        return "The index is invalid"
    show(completion_notes)
    return get_report_todo()

In [8]:
create_todo_list_json={
    "name":"create_todo_list",
    "description":"Use this tool to create list of steps that are needed to be execute and return full list",
    "parameters":{
        "type":"object",
        "properties":{
            "list_task":{
                "type":"array",
                "items":{"type":"string"},
                "description":"List of working steps"
            },
        },
        "required":["list_task"],
        "additionalProperties":False
        }
    }

In [9]:
mark_complete_json={
    "name":"mark_complete",
    "description":"Mark the step is completed using 1-bases index with completion feedback",
    "parameters":{
        "type":"object",
        "properties":{
            "index":{
                "type":"integer",
                "description":"1-bases index of completed step"
            },
            "completion_notes":{
                "type":"string",
                "description":"Consise completion feedback"
            },
        },
        "required":["index","completion_notes"],
        "additionalProperties":False
        }
    }

In [10]:
tools=[{"type":"function","function":create_todo_list_json},
    {"type":"function","function":mark_complete_json}]

In [11]:
def handle_tool_calls(tool_calls):
    results=[]
    for tool in tool_calls:
        tool_name=tool.function.name
        tool_arguments=json.loads(tool.function.arguments)
        function=globals().get(tool_name)
        result=function(**tool_arguments)
        results.append({"role":"tool","content":json.dumps(result),"tool_call_id":tool.id})
    return results

In [12]:
system_prompt="""You are given problem to solve, break the solution by steps using the provided tool, and execute it step by step.
Now using the todo list tools to create list of steps, carry out the steps, and reply with the solution.
The answer is concise with Rich console markup without code blocks.
Please do not ask user or reconfirm, just solve the problem
"""

user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

messages=[{"role":"system","content":system_prompt},{"role":"user","content":user_message}]

In [13]:
openai=OpenAI()

In [14]:
def run(messages):
    done=False
    while not done:
        response=openai.chat.completions.create(model="gpt-5.2",messages=messages,tools=tools,reasoning_effort="none")
        finish_reason=response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message=response.choices[0].message
            result=handle_tool_calls(message.tool_calls)
            messages.append(message)
            messages.extend(result)
        else:
            done=True
    show(response.choices[0].message.content)  

In [15]:
todo=[]
complete=[]
run(messages)

Step 1: Let t be hours after 2:00 pm when they meet; express distances traveled.
Step 2: Set up equation using total distance between Boston and New York (assume 300 miles) and relative motion.
Step 3: Solve for t and convert to clock time.
Step 4: Provide final meeting time.

Let t be hours after 2:00 pm. Boston train travels 60t miles. NY train leaves 1 hour later, so it travels 80(t−1) 
miles (for t≥1).

Step 1: Let t be hours after 2:00 pm when they meet; express distances traveled.
Step 2: Set up equation using total distance between Boston and New York (assume 300 miles) and relative motion.
Step 3: Solve for t and convert to clock time.
Step 4: Provide final meeting time.

Assume Boston–NY distance is 300 miles. Meeting condition: 60t + 80(t−1) = 300.

Step 1: Let t be hours after 2:00 pm when they meet; express distances traveled.
Step 2: Set up equation using total distance between Boston and New York (assume 300 miles) and relative motion.
Step 3: Solve for t and convert to clock time.
Step 4: Provide final meeting time.

Solve: 60t+80t−80=300 ⇒ 140t=380 ⇒ t=19/7≈2.714 hr after 2:00 pm = 2 hr 42 min 51 s.

Step 1: Let t be hours after 2:00 pm when they meet; express distances traveled.
Step 2: Set up equation using total distance between Boston and New York (assume 300 miles) and relative motion.
Step 3: Solve for t and convert to clock time.
Step 4: Provide final meeting time.

2:00 pm + 2:42:51 ≈ 4:42:51 pm, i.e., about 4:43 pm.

Step 1: Let t be hours after 2:00 pm when they meet; express distances traveled.
Step 2: Set up equation using total distance between Boston and New York (assume 300 miles) and relative motion.
Step 3: Solve for t and convert to clock time.
Step 4: Provide final meeting time.

Meeting time: about 4:43 pm

Let t = hours after 2:00 pm when they meet.  
Boston train: 60t miles.  
NY train leaves 1 hour later: 80(t−1) miles.  

Assuming Boston–New York is 300 miles:
60t + 80(t−1) = 300  
140t = 380 ⇒ t = 19/7 ≈ 2.714 hours = 2 h 42 m 51 s after 2:00 pm  
⇒ 4:42:51 pm (≈ 4:43 pm).